# Experiment 7 - Bagging, Boosting and Stacked Ensemble Models
Name: Gunaseelan R  
Roll No: 9342507779

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc

## 1. Load Dataset

In [ ]:
cols = ['id','diagnosis','radius_mean','texture_mean','perimeter_mean','area_mean','smoothness_mean',
'compactness_mean','concavity_mean','concave_points_mean','symmetry_mean','fractal_dimension_mean',
'radius_se','texture_se','perimeter_se','area_se','smoothness_se','compactness_se','concavity_se',
'concave_points_se','symmetry_se','fractal_dimension_se','radius_worst','texture_worst','perimeter_worst',
'area_worst','smoothness_worst','compactness_worst','concavity_worst','concave_points_worst',
'symmetry_worst','fractal_dimension_worst']

df = pd.read_csv('wdbc.data', header=None, names=cols)
df.head()

In [ ]:
df.drop('id', axis=1, inplace=True)
df.shape

## 2. EDA

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sum()

In [ ]:
df['diagnosis'].value_counts()

In [ ]:
sns.countplot(x='diagnosis', data=df)
plt.title('Class distribution')
plt.show()

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(df.drop('diagnosis',axis=1).corr(), cmap='coolwarm')
plt.title('Feature correlation')
plt.show()

## 3. Preprocessing

In [ ]:
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])   # M=1, B=0

X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

sc = StandardScaler()
X = sc.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape

## 4. Bagging Classifier

In [ ]:
bag_params = {
    'n_estimators': [10, 50, 100],
    'max_samples': [0.5, 0.7, 1.0]
}

bag = BaggingClassifier(estimator=DecisionTreeClassifier(), random_state=42)
bag_grid = GridSearchCV(bag, bag_params, cv=5, scoring='accuracy')
bag_grid.fit(X_train, y_train)

print(bag_grid.best_params_)
print(bag_grid.best_score_)

In [ ]:
bag_results = pd.DataFrame(bag_grid.cv_results_)[['param_n_estimators','param_max_samples','mean_test_score']]
bag_results

In [ ]:
bag_best = bag_grid.best_estimator_
bag_pred = bag_best.predict(X_test)

## 5. Boosting Classifiers

### AdaBoost

In [ ]:
ada_params = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 1]
}

ada = AdaBoostClassifier(random_state=42)
ada_grid = GridSearchCV(ada, ada_params, cv=5, scoring='accuracy')
ada_grid.fit(X_train, y_train)

print(ada_grid.best_params_)
print(ada_grid.best_score_)

### Gradient Boosting

In [ ]:
gb_params = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 1],
    'max_depth': [2, 3, 4]
}

gb = GradientBoostingClassifier(random_state=42)
gb_grid = GridSearchCV(gb, gb_params, cv=5, scoring='accuracy')
gb_grid.fit(X_train, y_train)

print(gb_grid.best_params_)
print(gb_grid.best_score_)

In [ ]:
# pick the better boosting model
if ada_grid.best_score_ >= gb_grid.best_score_:
    boost_best = ada_grid.best_estimator_
    print('AdaBoost selected')
else:
    boost_best = gb_grid.best_estimator_
    print('Gradient Boosting selected')

boost_pred = boost_best.predict(X_test)

## 6. Stacked Ensemble

In [ ]:
base_learners = [
    ('svm', SVC(probability=True)),
    ('nb', GaussianNB()),
    ('dt', DecisionTreeClassifier())
]

stack = StackingClassifier(estimators=base_learners, final_estimator=LogisticRegression(), cv=5)
stack.fit(X_train, y_train)
stack_pred = stack.predict(X_test)

stack_acc = accuracy_score(y_test, stack_pred)
stack_acc

## 7. Evaluation

In [ ]:
def get_metrics(name, y_true, y_pred):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred)
    }

results = pd.DataFrame([
    get_metrics('Bagging', y_test, bag_pred),
    get_metrics('Boosting', y_test, boost_pred),
    get_metrics('Stacked Ensemble', y_test, stack_pred)
])
results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
preds = {'Bagging': bag_pred, 'Boosting': boost_pred, 'Stacked Ensemble': stack_pred}

for ax, (name, pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,6))
models = {'Bagging': bag_best, 'Boosting': boost_best, 'Stacked Ensemble': stack}

for name, model in models.items():
    probs = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
results.plot(x='Model', y=['Accuracy','Precision','Recall','F1 Score'], kind='bar', figsize=(8,5))
plt.title('Model Comparison')
plt.ylim(0.8,1.0)
plt.show()

## 8. Observations

- Bagging reduces variance by training multiple decision trees on different bootstrap samples and averaging their predictions, which makes the model less sensitive to noise in the training data.
- Boosting reduces bias since each new model is trained to fix the mistakes of the previous ones, so the ensemble gradually gets better at hard examples.
- Stacking works well with heterogeneous base models (SVM, Naive Bayes, Decision Tree) because each model captures different patterns in the data, and the meta learner (Logistic Regression) learns how to combine their outputs optimally.
- From the results table, the Stacked Ensemble gave the best overall performance, followed by Boosting and Bagging.

## Conclusion
Bagging, Boosting and Stacked Ensemble models were implemented on the Wisconsin Diagnostic Breast Cancer dataset and compared using accuracy, precision, recall, F1-score, confusion matrix and ROC-AUC. The stacked ensemble performed the best since it combines the strengths of different types of classifiers.